# Dhruv Patel Academy blank-page practice: EDA & Data Cleaning

**Blank-page version:** every code cell is empty - write the whole answer yourself. (The fill-in-the-blank version is `eda.ipynb`.)

Practical questions from the **EDA & Data Cleaning** module. Write your answer in the code cell under each question, run it, then run the **check** cell right below it. The check runs *in this notebook*, on your own variables, and tells you exactly what is right or wrong.

1. Run the first code cell once (it loads the checker and the 6 questions).
2. Work through the questions in any order.
3. `dsq.hint("question-id")` gives a hint; `dsq.solution("question-id")` shows the reference answer (only after you tried 3 times).
4. `dsq.score()` shows where you stand.

Optional: `dsq.connect("http://127.0.0.1:5177", "YOUR-KEY")` (key from the module page in Dhruv Patel Academy) sends solved questions to the app so your profile counts them.

You need Python with `numpy`, `pandas`, `matplotlib` and `seaborn`, e.g. `pip install numpy pandas matplotlib seaborn notebook`.

In [ ]:
# --- Dhruv Patel Academy practice checker (runs in YOUR notebook, on YOUR variables) ---
import json as _json, sys as _sys, urllib.request as _ur

class _DSQ:
    def __init__(self):
        self.qs = {}; self.order = []; self.tries = {}; self.ok = {}; self.url = None; self.key = None; self.module = ''

    def load(self, pack):
        self.module = pack['title']
        for q in pack['questions']:
            self.qs[q['id']] = q; self.order.append(q['id'])
        print(f"Dhruv Patel Academy: {len(self.order)} questions loaded for '{self.module}'. Write your answer, then run the check cell under it.")
        if self._auto(): print('Linked to your Dhruv Patel Academy account: solved questions count in the app.')

    def connect(self, url, key, quiet=False):
        self.url = url.rstrip('/'); self.key = key
        if not quiet: print('Connected: solved questions will be reported to', self.url)

    def _auto(self):
        # Dhruv Patel Academy writes dsq_config.json next to the notebooks when you open them from the app
        import os
        d = os.getcwd()
        for _ in range(4):
            p = os.path.join(d, 'dsq_config.json')
            if os.path.exists(p):
                try:
                    c = _json.load(open(p)); self.connect(c['url'], c['key'], quiet=True); return True
                except Exception:
                    pass
            d = os.path.dirname(d)
        return False

    def _report(self, qid, ok):
        if not self.url: return
        try:
            req = _ur.Request(self.url + '/api/nb/report', data=_json.dumps({'key': self.key, 'qid': qid, 'ok': bool(ok)}).encode(), headers={'Content-Type': 'application/json'})
            _ur.urlopen(req, timeout=4).read()
        except Exception as e:
            print('(could not reach the app:', str(e)[:80] + ')')

    def check(self, qid):
        q = self.qs[qid]
        try:
            ip = get_ipython(); ns = dict(ip.user_ns)
        except NameError:
            ns = dict(_sys.modules['__main__'].__dict__)
        results = []
        def check(cond, msg, hint=''):
            ok = bool(cond); results.append((ok, msg, hint)); return ok
        ns['check'] = check; ns['OUT'] = ''
        for mod, alias in (('numpy', 'np'), ('pandas', 'pd')):
            if alias not in ns:
                try: ns[alias] = __import__(mod)
                except Exception: pass
        if 'plt' not in ns:
            try:
                import matplotlib.pyplot as plt; ns['plt'] = plt
            except Exception: pass
        try:
            exec(compile(q['tests'], '<tests>', 'exec'), ns)
        except BaseException as e:
            results.append((False, 'A check could not finish: ' + type(e).__name__ + ': ' + str(e)[:200], 'Did you create every variable name the task asks for? Did you run your answer cell first?'))
        self.tries[qid] = self.tries.get(qid, 0) + 1
        good = bool(results) and all(r[0] for r in results)
        for ok, msg, hint in results:
            print(('  [PASS] ' if ok else '  [FAIL] ') + msg + ('' if ok or not hint else '\n         hint: ' + hint))
        if good:
            self.ok[qid] = True; print(f"\n  Question solved: {q['title']}  (attempt {self.tries[qid]})")
        else:
            print(f"\n  Not yet - fix your answer cell, run it again, then re-run this check. (attempt {self.tries[qid]})")
        self._report(qid, good)
        return good

    def hint(self, qid):
        hs = self.qs[qid].get('hints') or []
        n = min(self.tries.get(qid, 0), len(hs)) if hs else 0
        print(('hint: ' + hs[max(0, n - 1)]) if hs else 'No hint for this one - re-read the question.')

    def solution(self, qid):
        if self.tries.get(qid, 0) < 3 and not self.ok.get(qid):
            print('Try the check at least 3 times first - the struggle is where the learning happens.'); return
        print(self.qs[qid]['solution'])

    def score(self):
        n = len(self.order); s = sum(1 for q in self.order if self.ok.get(q))
        print(f"Solved {s} of {n}.")
        for q in self.order:
            print(('  [x] ' if self.ok.get(q) else '  [ ] ') + q + ' - ' + self.qs[q]['title'] + ('' if self.ok.get(q) else f"  (tries: {self.tries.get(q, 0)})"))

dsq = _DSQ()


dsq.load({
 "module": "eda",
 "title": "EDA & Data Cleaning",
 "questions": [
  {
   "id": "eda-1-1",
   "title": "Write the audit",
   "tests": "check(n_rows == 7 and n_dupes == 0, '7 rows and no exactly repeated rows (the two \"61000\" rows differ in city case, so they are not yet exact repeats)', 'len(df), df.duplicated().sum()')\ncheck(missing.to_dict() == {'age': 1, 'city': 0, 'income': 0}, 'missing: age 1 (the \"n/a\" in income is TEXT, not NaN yet!)', 'df.isna().sum()')\ncheck(worst_col == 'age' and sorted(text_cols) == ['city', 'income'], 'worst_col = age; text columns = city, income', 'a list comprehension over df.columns')",
   "hints": [
    "df.duplicated().sum() counts repeated rows.",
    "idxmax on the missing counts gives the column name.",
    "df[c].dtype == object tests for text columns."
   ],
   "solution": "import pandas as pd\nimport numpy as np\n\ndf = pd.DataFrame({\n    \"age\":    [25, 31, np.nan, 29, 187, 31, 40],\n    \"city\":   [\" Pune\", \"delhi\", \"PUNE\", \"Delhi \", \"Pune\", \"DELHI\", \"Mumbai\"],\n    \"income\": [\"52,000\", \"61000\", \"n/a\", \"58,500\", \"49000\", \"61000\", \"910,000\"],\n})\n\nn_rows = len(df)\nn_dupes = int(df.duplicated().sum())\nmissing = df.isna().sum()\nworst_col = missing.idxmax()\ntext_cols = [c for c in df.columns if df[c].dtype == object]\nprint(n_rows, n_dupes, worst_col, text_cols)"
  },
  {
   "id": "eda-2-1",
   "title": "Clean the text and the numbers",
   "tests": "check(str(df['income'].dtype).startswith('float') and df['income'].isna().sum() == 1, 'income is numeric with exactly one NaN (the \"n/a\")', 'to_numeric(..., errors=\"coerce\")')\ncheck(sorted(df['city'].unique()) == ['Delhi', 'Mumbai', 'Pune'], 'cities are Delhi, Mumbai, Pune', '.str.strip().str.title()')\ncheck(len(df) == 6, '6 rows remain: the repeated customer only shows up once the text is tidy', 'drop_duplicates() AFTER tidying the text')",
   "hints": [
    "Remove the comma with .str.replace(\",\", \"\") before to_numeric.",
    "Rows 2 and 5 only become identical after the city text is tidy."
   ],
   "solution": "import pandas as pd\nimport numpy as np\n\ndf = pd.DataFrame({\n    \"age\":    [25, 31, np.nan, 29, 187, 31, 40],\n    \"city\":   [\" Pune\", \"delhi\", \"PUNE\", \"Delhi \", \"Pune\", \"DELHI\", \"Mumbai\"],\n    \"income\": [\"52,000\", \"61000\", \"n/a\", \"58,500\", \"49000\", \"61000\", \"910,000\"],\n})\n\ndf[\"income\"] = pd.to_numeric(df[\"income\"].astype(str).str.replace(\",\", \"\"), errors=\"coerce\")\ndf[\"city\"] = df[\"city\"].str.strip().str.title()\ndf = df.drop_duplicates()\nprint(df)"
  },
  {
   "id": "eda-2-2",
   "title": "The IQR outlier rule, by hand",
   "tests": "_q1, _q3 = np.percentile(v, 25), np.percentile(v, 75); _i = _q3 - _q1\ncheck(abs(q1 - _q1) < 1e-9 and abs(q3 - _q3) < 1e-9 and abs(iqr - _i) < 1e-9, 'q1, q3 and iqr are right', 'np.percentile(v, 25), np.percentile(v, 75)')\ncheck(abs(lo - (_q1 - 1.5 * _i)) < 1e-9 and abs(hi - (_q3 + 1.5 * _i)) < 1e-9, 'the fences are Q1 - 1.5 IQR and Q3 + 1.5 IQR', 'q1 - 1.5 * iqr, q3 + 1.5 * iqr')\ncheck(int(is_out.sum()) == 1 and v[is_out][0] == 910000 and clean.size == 8, 'only 910000 is an outlier; 8 values remain', '(v < lo) | (v > hi)')",
   "hints": [
    "np.percentile(v, 25) or np.quantile(v, 0.25).",
    "Combine two conditions with | and invert a mask with ~."
   ],
   "solution": "import numpy as np\n\nv = np.array([52000, 61000, 58500, 49000, 61000, 910000, 55000, 47000, 63000])\nq1 = np.percentile(v, 25)\nq3 = np.percentile(v, 75)\niqr = q3 - q1\nlo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr\nis_out = (v < lo) | (v > hi)\nclean = v[~is_out]\nprint(lo, hi, is_out.sum(), clean.size)"
  },
  {
   "id": "eda-3-1",
   "title": "Spot the redundant column",
   "tests": "_c = np.corrcoef(X.T)\ncheck(corr.shape == (4, 4) and np.allclose(corr, _c), 'corr is the 4x4 correlation matrix', 'np.corrcoef(X.T)')\ncheck(abs(det) < 1e-6 and abs(det - np.linalg.det(_c)) < 1e-9, 'the determinant is ~0: the columns are redundant', 'np.linalg.det(corr)')\ncheck(int(rank) == 3 and drop == 2, 'rank 3 (not 4); drop column 2 (the thousand-times copy)', 'np.linalg.matrix_rank(X); the later column of the correlated pair')",
   "hints": [
    "np.corrcoef expects variables in ROWS, so pass X.T.",
    "np.triu(mask, k=1) keeps only the part above the diagonal so each pair appears once."
   ],
   "solution": "import numpy as np\n\nrng = np.random.default_rng(1)\na = rng.normal(55, 12, 200)\nX = np.column_stack([a, rng.normal(40, 8, 200), a * 1000, rng.normal(0, 1, 200)])\ncorr = np.corrcoef(X.T)\ndet = np.linalg.det(corr)\nrank = np.linalg.matrix_rank(X)\ni, j = np.argwhere(np.triu(np.abs(corr) > 0.99, k=1))[0]\ndrop = int(j)\nprint(round(det, 6), rank, drop)"
  },
  {
   "id": "eda-3-2",
   "title": "Distance from the centre (Mahalanobis)",
   "tests": "_mu = X.mean(axis=0); _cv = np.cov(X.T); _iv = np.linalg.inv(_cv); _df = X - _mu; _d = np.sqrt(np.einsum('ij,jk,ik->i', _df, _iv, _df))\ncheck(np.allclose(mu, _mu) and np.allclose(cov, _cv) and np.allclose(inv, _iv), 'mu, cov and inv are right', 'X.mean(axis=0), np.cov(X.T), np.linalg.inv(cov)')\ncheck(np.allclose(d, _d) and d.shape == (150,), 'd holds the Mahalanobis distance of every row', 'np.sqrt(einsum(...))')\ncheck(worst == 0 and abs(X[0, 0]) < 2 and abs(X[0, 1]) < 2, 'row 0 is the biggest outlier although neither value is extreme on its own', 'np.argmax(d)')",
   "hints": [
    "One row: np.sqrt(diff[i] @ inv @ diff[i]). For all rows at once np.einsum(\"ij,jk,ik->i\", diff, inv, diff) does it without a loop."
   ],
   "solution": "import numpy as np\n\nrng = np.random.default_rng(4)\nx1 = rng.normal(0, 1, 150)\nX = np.column_stack([x1, x1 + rng.normal(0, 0.15, 150)])\nX[0] = [1.5, -1.5]          # fine on each column alone, but x2 should follow x1!\nmu = X.mean(axis=0)\ncov = np.cov(X.T)\ninv = np.linalg.inv(cov)\ndiff = X - mu\nd = np.sqrt(np.einsum(\"ij,jk,ik->i\", diff, inv, diff))\nworst = int(np.argmax(d))\nprint(round(d[0], 2), worst)"
  },
  {
   "id": "eda-4-1",
   "title": "Cleaning log + before/after chart",
   "tests": "fig = plt.gcf(); a_, b_ = fig.axes[:2]\ncheck(len(fig.axes) == 2 and a_.get_title() == 'before' and b_.get_title() == 'after', 'two charts titled \"before\" and \"after\"', 'a.set_title(\"before\")')\ncheck(a_.get_ylabel() and a_.get_ylabel() != '___', 'the y axis has a real label', 'a.set_ylabel(\"customers\")')\ncheck(len(log) == 3 and '3' in log[0] and '2' in log[1] and '7' in log[2], 'the log has the right counts (3 duplicates, 2 outliers, 7 left)', 'f-strings with n1, n2, len(df)')",
   "hints": [
    "The left chart shows the data before cleaning."
   ],
   "solution": "import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nraw = pd.DataFrame({\"income\": [52, 61, 58, 49, 61, 910, 55, 47, 63, 61, 58, 700]})\ndf = raw.drop_duplicates()\nn1 = len(raw) - len(df)\nq1, q3 = df[\"income\"].quantile([0.25, 0.75])\ndf = df[df[\"income\"] <= q3 + 1.5 * (q3 - q1)]\nn2 = len(raw) - n1 - len(df)\nlog = [f\"dropped {n1} duplicates\", f\"removed {n2} outliers\", f\"{len(df)} rows left\"]\n\nfig, (a, b) = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)\na.hist(raw[\"income\"], bins=10)\na.set_title(\"before\")\nb.hist(df[\"income\"], bins=10)\nb.set_title(\"after\")\na.set_ylabel(\"customers\")"
  }
 ]
})


## 1. Write the audit  `eda-1-1`
*(from the lesson: Audit first: what is wrong with this data?)*

Compute: `n_rows`, `n_dupes` (duplicated rows), `missing` (missing count per column, a Series), `worst_col` (the column with the most missing values) and `text_cols`: a **list** of the columns whose dtype is `object`.

In [ ]:
# your code here


In [ ]:
dsq.check("eda-1-1")

## 2. Clean the text and the numbers  `eda-2-1`
*(from the lesson: Fixing it: text, types, duplicates, missing, outliers)*

Fix `df` in this order: (1) `income` to real numbers (remove commas, unreadable -> NaN), (2) tidy `city` (strip + Title Case), (3) drop duplicate rows. Keep the result in `df`.

In [ ]:
# your code here


In [ ]:
dsq.check("eda-2-1")

## 3. The IQR outlier rule, by hand  `eda-2-2`
*(from the lesson: Fixing it: text, types, duplicates, missing, outliers)*

`v` holds incomes. Compute `q1`, `q3`, `iqr`, the fences `lo` and `hi` (1.5 x IQR rule), `is_out` (boolean mask of outliers) and `clean` (the values inside the fences).

In [ ]:
# your code here


In [ ]:
dsq.check("eda-2-2")

## 4. Spot the redundant column  `eda-3-1`
*(from the lesson: Matrices in cleaning: correlation, determinant, distance)*

Compute `corr` (correlation matrix of the columns of `X`), `det` (its determinant), `rank` (matrix rank of `X`) and `drop`: the **index** of the LATER column of the one pair with |correlation| above 0.99.

In [ ]:
# your code here


In [ ]:
dsq.check("eda-3-1")

## 5. Distance from the centre (Mahalanobis)  `eda-3-2`
*(from the lesson: Matrices in cleaning: correlation, determinant, distance)*

Row 0 of `X` looks ordinary on each column but breaks the correlation. Compute `mu` (column means), `cov` (covariance matrix, `np.cov(X.T)`), `inv` (its inverse) and `d`: the Mahalanobis distance of EVERY row, `sqrt((x - mu) @ inv @ (x - mu))`. `worst` = index of the largest distance.

In [ ]:
# your code here


In [ ]:
dsq.check("eda-3-2")

## 6. Cleaning log + before/after chart  `eda-4-1`
*(from the lesson: Show your cleaning: before and after)*

Build `log`: a list of 3 strings describing how many rows each step removed (`"dropped 2 duplicates"`, etc. - any wording, but it must contain the correct number). Then draw before/after histograms of `income` side by side (`raw` vs `df`), titled "before" and "after" with a y label.

In [ ]:
# your code here


In [ ]:
dsq.check("eda-4-1")

## Done?

In [ ]:
dsq.score()